In [ ]:
# ==========================================
# 0. 【大絕招】自動檢查並安裝缺少的工具箱
# ==========================================
import sys
import subprocess

# 檢查必備套件，如果沒有就自動在 Jupyter 內部下載安裝
required_packages = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn"]
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Detecting missing package: {package}. Installing now...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("All required toolboxes are successfully installed and loaded!")

# ==========================================
# 1. 套件正式匯入與資料讀取
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# 讀取資料（請確保 train.csv 檔案跟你們的 ipynb 放在同一個資料夾）
try:
    df = pd.read_csv('train.csv')
    print(f"Data loaded successfully! Total records: {df.shape[0]} passengers.")
except FileNotFoundError:
    print("【錯誤提示】找不到 train.csv 檔案！請確保鐵達尼號的原始資料 train.csv 與此程式碼放在同一個資料夾喔！")

# 2. 探索性資料分析 (EDA) 視覺化
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

#Survival Rate by Sex
sns.barplot(x='Sex', y='Survived', data=df, ax=axes[0], palette='pastel', ci=None)
axes[0].set_title('Survival Rate by Sex')
axes[0].set_ylabel('Survival Rate')

#Survival Rate by Pclass
sns.barplot(x='Pclass', y='Survived', data=df, ax=axes[1], palette='muted', ci=None)
axes[1].set_title('Survival Rate by Pclass')
axes[1].set_ylabel('Survival Rate')
plt.show()

#Correlation Heatmap - 僅限數值型欄位
plt.figure(figsize=(8, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Heatmap')
plt.show()

# 剔除與生存無直接邏輯關聯的流水號與文字欄位
df_clean = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# 分離特徵與標籤
X = df_clean.drop(columns=['Survived'])
y = df_clean['Survived']

# 第一時間切分訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 將 Pclass 視為類別變數，與 Sex, Embarked 一起做 One-Hot Encoding
# 加上 drop_first=True 移除基準面，避免虛擬變數陷阱
X_train = pd.get_dummies(X_train, columns=['Pclass', 'Sex', 'Embarked'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['Pclass', 'Sex', 'Embarked'], drop_first=True)

# 確保訓練集與測試集的欄位完全對齊
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

# 缺失值填充
imputer = SimpleImputer(strategy='median')
X_train['Age'] = imputer.fit_transform(X_train[['Age']])
X_test['Age'] = imputer.transform(X_test[['Age']]) # Test 只能 transform

# 數值特徵標準化
scaler = StandardScaler()
X_train[['Age', 'Fare']] = scaler.fit_transform(X_train[['Age', 'Fare']])
X_test[['Age', 'Fare']] = scaler.transform(X_test[['Age', 'Fare']]) # Test 只能 transform

print(f"Preprocessing completed! Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# 模型訓練與評估
# 建立隨機森林分類器
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# 進行預測
y_pred = rf_model.predict(X_test)

print("\n" + "="*30)
print("=== Classification Report ===")
print("="*30)
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Died', 'Survived'])
disp.plot(ax=ax1, cmap='Blues', values_format='d')
ax1.set_title('Confusion Matrix')
ax1.grid(False) # 移除背景格線讓圖表更乾淨

# Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X_train.columns

sns.barplot(x=importances[indices], y=features[indices], ax=ax2, palette='viridis')
ax2.set_title('Feature Importance (Random Forest)')
ax2.set_xlabel('Importance Score')
ax2.set_ylabel('Features')

plt.tight_layout()
plt.show()

: 